# Module 4 — Functions

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Define functions with parameters and return values, and document them with docstrings and type hints.
2. Use positional, keyword, default, `*args` and `**kwargs` arguments.
3. Understand **scope** (where variables live) and why side effects are dangerous.
4. Treat functions as values: pass them to other functions, use `lambda`, `map`, `filter`, `sorted(key=...)`.
5. Write simple recursive functions.

---

## 4.1 Why functions?

A function is a **named, reusable block of code**. Functions let you:

* avoid repeating yourself (the *DRY* principle — Don't Repeat Yourself);
* break a big problem into small, testable pieces;
* give a meaningful name to a computation (`mean_squared_error(y, y_hat)` is clearer than a raw formula).

## 4.2 Defining and calling a function

In [1]:
def greet(name):
    """Return a greeting for `name`."""      # docstring: documents the function
    return f"Hello, {name}!"

message = greet("Swansea")
print(message)

Hello, Swansea!


Anatomy:

```text
def  function_name ( parameters ) :
    """docstring"""
    body
    return value
```

* `def` starts the definition; the body is indented.
* **Parameters** are the names in the definition; **arguments** are the values you pass when calling.
* `return` sends a value back **and ends the function**. A function without `return` returns `None`.

In [1]:
def say_hi():
    print("hi")          # prints, but returns nothing

result = say_hi()
print("returned:", result)

hi
returned: None


> **print vs return:** `print` only shows a value on screen; `return` hands it back so that the rest of your program can use it. Functions that compute something should almost always `return`.

### A first ML function: mean squared error

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat y_i)^2$$

In [2]:
def mse(y_true, y_pred):
    """Mean squared error between two equal-length sequences."""
    n = len(y_true)
    total = 0.0
    for t, p in zip(y_true, y_pred):
        total += (t - p) ** 2
    return total / n

print(mse([3.0, 5.0, 2.5], [2.5, 5.0, 3.0]))

0.16666666666666666


### Returning several values

Return a tuple and unpack it at the call site:

In [4]:
def min_max_mean(values):
    return min(values), max(values), sum(values) / len(values)

lo, hi, avg = min_max_mean([4, 8, 15, 16, 23, 42])
print(lo, hi, avg)

4 42 18.0


## 4.3 Type hints and docstrings

**Type hints** state what types a function expects and returns. Python does *not* enforce them at run time, but they make code self-documenting and allow editors and tools (e.g. `mypy`) to catch bugs.

In [5]:
def accuracy(y_true: list[int], y_pred: list[int]) -> float:
    """Compute classification accuracy.

    Parameters
    ----------
    y_true : list[int]
        Ground-truth labels.
    y_pred : list[int]
        Predicted labels, same length as ``y_true``.

    Returns
    -------
    float
        Fraction of correct predictions, between 0 and 1.
    """
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

print(accuracy([1, 0, 1, 1], [1, 1, 1, 0]))
help(accuracy)

0.5
Help on function accuracy in module __main__:

accuracy(y_true: list[int], y_pred: list[int]) -> float
    Compute classification accuracy.

    Parameters
    ----------
    y_true : list[int]
        Ground-truth labels.
    y_pred : list[int]
        Predicted labels, same length as ``y_true``.

    Returns
    -------
    float
        Fraction of correct predictions, between 0 and 1.



The docstring layout above is the **NumPy style**, used by NumPy, pandas and scikit-learn. Reading those libraries' documentation will feel familiar.

## 4.4 Kinds of arguments

### Positional and keyword arguments

In [6]:
def describe(name, epochs, lr):
    return f"{name}: {epochs} epochs, lr={lr}"

print(describe("CNN", 10, 0.001))                 # positional: order matters
print(describe(lr=0.001, name="CNN", epochs=10))  # keyword: order does not matter
print(describe("CNN", lr=0.001, epochs=10))       # mixed: positional ones must come first

CNN: 10 epochs, lr=0.001
CNN: 10 epochs, lr=0.001
CNN: 10 epochs, lr=0.001


### Default values

Parameters with defaults become optional. This is exactly how scikit-learn models work: `RandomForestClassifier()` uses sensible defaults, and you override only what you need.

In [7]:
def train(model, epochs=10, lr=0.01, verbose=False):
    msg = f"training {model} for {epochs} epochs (lr={lr})"
    if verbose:
        msg += " [verbose]"
    return msg

print(train("MLP"))
print(train("MLP", epochs=50))
print(train("MLP", lr=0.1, verbose=True))

training MLP for 10 epochs (lr=0.01)
training MLP for 50 epochs (lr=0.01)
training MLP for 10 epochs (lr=0.1) [verbose]


### ⚠️ The mutable default argument trap

Default values are created **once**, when the function is defined — not each time it is called. A mutable default (list, dict) is therefore shared between calls:

In [8]:
def add_score_bad(score, history=[]):
    history.append(score)
    return history

print(add_score_bad(1))
print(add_score_bad(2))   # surprise: the previous score is still there!

def add_score_good(score, history=None):
    if history is None:
        history = []      # a fresh list on every call
    history.append(score)
    return history

print(add_score_good(1))
print(add_score_good(2))

[1]
[1, 2]
[1]
[2]


**Rule:** use `None` as the default for mutable parameters.

### `*args` and `**kwargs`

* `*args` collects extra **positional** arguments into a tuple.
* `**kwargs` collects extra **keyword** arguments into a dictionary.

In [9]:
def average(*args):
    return sum(args) / len(args)

print(average(1, 2, 3))
print(average(10, 20))

def build_model(name, **kwargs):
    print("model:", name)
    for k, v in kwargs.items():
        print(f"  {k} = {v}")

build_model("SVM", C=1.0, kernel="rbf", gamma="scale")

2.0
15.0
model: SVM
  C = 1.0
  kernel = rbf
  gamma = scale


The `*` and `**` operators also **unpack** sequences and dictionaries when calling a function — very common for passing a configuration dictionary:

In [10]:
values = [3, 4, 5]
print(average(*values))           # same as average(3, 4, 5)

config = {"epochs": 30, "lr": 0.05}
print(train("ResNet", **config))  # same as train("ResNet", epochs=30, lr=0.05)

4.0
training ResNet for 30 epochs (lr=0.05)


### Keyword-only arguments

Parameters after a bare `*` **must** be given by keyword. Libraries use this to make calls unambiguous:

In [11]:
def split(data, *, test_size=0.2, shuffle=True):
    n_test = int(len(data) * test_size)
    return data[:-n_test], data[-n_test:]

train_part, test_part = split(list(range(10)), test_size=0.3)
print(train_part, test_part)

[0, 1, 2, 3, 4, 5, 6] [7, 8, 9]


## 4.5 Scope: where variables live

Variables created inside a function are **local**: they exist only while the function runs. Python looks names up in the order **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in (**LEGB**).

In [12]:
threshold = 0.5           # global

def classify(p):
    label = 1 if p >= threshold else 0   # reads the global 'threshold'
    return label

print(classify(0.7))
try:
    print(label)          # 'label' only existed inside the function
except NameError as e:
    print("NameError:", e)

1
NameError: name 'label' is not defined


Assigning to a name inside a function creates a **new local variable**; it does not change the global one:

In [13]:
counter = 0

def increment():
    counter = 100        # local variable, unrelated to the global one
    return counter

increment()
print(counter)           # still 0

0


You *can* modify a global with the `global` keyword, but this is usually bad design. Prefer **pure functions**: they take inputs as parameters, return outputs, and do not modify anything outside themselves. Pure functions are easy to test and to reason about.

### Side effects on mutable arguments

A function receives a **reference** to the object you pass (Module 2). If it modifies a mutable argument, the caller sees the change:

In [14]:
def normalise_in_place(values):
    total = sum(values)
    for i in range(len(values)):
        values[i] /= total       # modifies the caller's list!

def normalised(values):
    total = sum(values)
    return [v / total for v in values]   # returns a new list instead

data = [1, 2, 7]
print(normalised(data), data)
normalise_in_place(data)
print(data)

[0.1, 0.2, 0.7] [1, 2, 7]
[0.1, 0.2, 0.7]


Both styles exist in libraries (pandas has `inplace=` options; NumPy has `out=`), but returning a new object is the safer default.

## 4.6 Functions are objects

In Python, functions are values: you can store them in variables, put them in lists or dictionaries, and pass them to other functions.

In [15]:
def relu(x):
    return max(0.0, x)

def identity(x):
    return x

activations = {"relu": relu, "identity": identity}
f = activations["relu"]
print(f(-2.0), f(3.0))

def apply_all(func, values):
    return [func(v) for v in values]

print(apply_all(relu, [-1, 0, 2, -3, 5]))

0.0 3.0
[0.0, 0.0, 2, 0.0, 5]


### `lambda`: small anonymous functions

`lambda arguments: expression` creates a one-line function without a name. Use it for short functions passed as arguments.

In [16]:
square = lambda x: x ** 2      # works, but a 'def' is preferred when you name it
print(square(5))

results = [("kNN", 0.81), ("SVM", 0.93), ("Tree", 0.77)]
print(sorted(results, key=lambda pair: pair[1], reverse=True))   # sort by score
print(max(results, key=lambda pair: pair[1]))

25
[('SVM', 0.93), ('kNN', 0.81), ('Tree', 0.77)]
('SVM', 0.93)


### `map` and `filter`

In [17]:
nums = [1, 2, 3, 4, 5, 6]
print(list(map(lambda x: x * 10, nums)))
print(list(filter(lambda x: x % 2 == 0, nums)))
# Comprehensions are usually clearer:
print([x * 10 for x in nums], [x for x in nums if x % 2 == 0])

[10, 20, 30, 40, 50, 60]
[2, 4, 6]
[10, 20, 30, 40, 50, 60] [2, 4, 6]


### Closures and function factories

A function defined inside another function can remember variables from the enclosing scope:

In [18]:
def make_scaler(factor):
    def scale(x):
        return x * factor     # 'factor' is remembered from the outer function
    return scale

double = make_scaler(2)
to_percent = make_scaler(100)
print(double(21), to_percent(0.37))

42 37.0


## 4.7 Recursion

A recursive function calls itself on a smaller version of the problem. It needs a **base case** that stops the recursion.

In [19]:
def factorial(n: int) -> int:
    if n <= 1:            # base case
        return 1
    return n * factorial(n - 1)   # recursive case

print(factorial(5))

120


Recursion is natural for tree-like structures (e.g. decision trees, nested JSON). Python limits recursion depth (around 1000 calls), so for long simple repetitions a loop is better.

In [20]:
def depth(nested):
    """Maximum nesting depth of a list."""
    if not isinstance(nested, list):
        return 0
    if not nested:
        return 1
    return 1 + max(depth(item) for item in nested)

print(depth([1, [2, [3, [4]]], 5]))

4


## 4.8 Good practice

* One function, one job. If you need "and" to describe it, split it.
* Choose descriptive names (`compute_precision`, not `cp`).
* Write a docstring and type hints for anything you will reuse.
* Avoid global variables and hidden side effects.
* Test functions with small, known inputs (`assert` is a quick way):

In [21]:
assert mse([1, 2], [1, 2]) == 0
assert abs(mse([0, 0], [1, 1]) - 1.0) < 1e-12
print("all tests passed")

all tests passed


---

## Summary

* `def name(params): ... return value`. No `return` means `None`.
* Defaults make parameters optional; never use a mutable default — use `None`.
* `*args` / `**kwargs` collect extra arguments; `*` / `**` unpack them when calling.
* Local variables live only inside the function (LEGB rule). Prefer pure functions.
* Functions are objects: pass them around, use `lambda` for short ones, `key=` for sorting.

---

## Exercises

Solutions: `solutions/04_functions_solutions.ipynb`.

**Exercise 4.1 — Statistics functions.**
Write `mean(values)`, `variance(values)` (population variance: $\frac{1}{n}\sum (x_i-\bar x)^2$) and `std(values)` (square root of the variance). `std` should **call** `variance`, which should call `mean`. Add docstrings and type hints. Test on `[2, 4, 4, 4, 5, 5, 7, 9]` (mean 5, std 2).

In [22]:
# Your code here

**Exercise 4.2 — Min-max scaling.**
Write `min_max_scale(values, new_min=0.0, new_max=1.0)` returning a **new** list where each value is mapped linearly so that the smallest becomes `new_min` and the largest becomes `new_max`:
$x' = \text{new\_min} + \frac{x - \min}{\max - \min}(\text{new\_max} - \text{new\_min})$.
If all values are equal, return a list filled with `new_min`. Test with defaults and with `new_min=-1, new_max=1`.

In [23]:
# Your code here

**Exercise 4.3 — Flexible metric function.**
Write `evaluate(y_true, y_pred, *metrics)` where each metric is a function `(y_true, y_pred) -> float`. It should return a dictionary `{metric_name: value}` (hint: a function's name is `func.__name__`). Test it with `mse` and `mae` (mean absolute error), which you also write.

In [24]:
# Your code here

**Exercise 4.4 — Find the bug.**
The function below is supposed to return a **new** list with each sample's label appended, but repeated calls give strange results and the input list is changed. Explain the two problems in comments and fix them.

In [25]:
def add_label(sample, label, out=[]):
    sample.append(label)
    out.append(sample)
    return out

s1 = [5.1, 3.5]
print(add_label(s1, "setosa"))
print(add_label([6.2, 2.9], "versicolor"))
print(s1)

[[5.1, 3.5, 'setosa']]
[[5.1, 3.5, 'setosa'], [6.2, 2.9, 'versicolor']]
[5.1, 3.5, 'setosa']


**Exercise 4.5 — Sorting with `key`.**
Using `sorted` with a `lambda`: (a) sort `models` by score, highest first; (b) sort by training time, lowest first; (c) sort by name length and then alphabetically (hint: the key can return a tuple).

In [26]:
models = [("SVM", 0.91, 3.2), ("kNN", 0.85, 0.4), ("RandomForest", 0.93, 8.1), ("LogReg", 0.88, 0.9)]
# each item: (name, score, training_time_seconds)
# Your code here

**Exercise 4.6 — Function factory.**
Write `make_polynomial(*coefficients)` that returns a function `p(x)` computing $c_0 + c_1x + c_2x^2 + \dots$. For example, `make_polynomial(1, 0, 2)` returns $p(x) = 1 + 2x^2$, so `p(3) == 19`.

In [27]:
# Your code here

**Exercise 4.7 — Recursion (challenge).**
(a) Write a recursive `fibonacci(n)` (with `fibonacci(0) = 0`, `fibonacci(1) = 1`). (b) Write a recursive `flatten(nested)` that turns `[1, [2, [3, 4], 5], [[6]]]` into `[1, 2, 3, 4, 5, 6]`.

In [28]:
# Your code here